# 김희서 코드 새로운 버전0905 — GN 3편 + Petrovich HCF 독립 검증

기준 코드는 `paper/hcf-optimum-launch-power-reproduction/dnanf_fig2_hcf_colab.ipynb`의
HCF Fig. 2 물리 모델을 보존한다. 기존 Fig. 2 재현 결과와 네 연구 항목의 독립 검증을
분리하여 실행한다.

1. Poggiolini (JLT 2012): GN closed-form 구현과 Eq. 13 일관성 점검.
2. Nespola et al. (PM-16QAM, 7 fiber): Table I의 입력으로 실험 Lmax를 재계산.
3. Carena et al. (JLT 2012): 첨부 그림의 정확한 조건이 확보되지 않은 부분은 계산하지 않고 입력 인터페이스만 제공.
4. Petrovich et al. (Nature Photonics 2025): Raw Hasan/MS 분산과 FEM, raw bouncing-ray proxy와 측정 총손실을 물리적 층으로 구분해 비교.

참조 출력값은 실행 뒤 오차 계산에만 사용하며 모델 파라미터에 피팅하지 않는다.

주요 출처: [Poggiolini 저자 원문](https://iris.polito.it/handle/11583/2506445),
[Nespola 공개 PDF](https://backoffice.biblio.ugent.be/download/5743918/5743952),
[Carena 논문 초록](https://opg.optica.org/abstract.cfm?uri=jlt-30-10-1524),
[Petrovich preprint](https://arxiv.org/abs/2503.21467).


# Fig. 2 HCF 처리량 — 물리 기반 Colab 재현

이 노트북은 Sohanpal *et al.*의 Fig. 2 중 **HCF(1×200 km) 곡선만** 계산합니다. 그림의 좌표를 추출하거나 처리량 값을 임의로 배열에 넣지 않습니다.

각 C-band 채널에 대해 다음 모델을 직접 계산합니다.

$$
\mathrm{SNR}_i = \frac{P_i}{P_{\mathrm{ASE},i}+\eta_iP_i^3+P_{\mathrm{IMI},i}+P_{\mathrm{TRN},i}},
\qquad
T_i=2R\log_2(1+\mathrm{SNR}_i).
$$

총 처리량은 29개 채널을 합한 $T_{\rm tot}=\sum_iT_i$입니다.

- **ASE:** 채널별 DNANF 손실과 5 dB 증폭기 NF에서 계산
- **NLI:** closed-form Gaussian-noise(GN) 모델로 $\eta_i$ 계산
- **IMI:** 논문의 $-52$ dB/km 계수 사용
- **송수신기 잡음:** 논문의 20 dB SNR 한계 사용
- **손실·분산:** Fig. 1용 연속 DNANF 물리 모델을 C-band 29개 채널에서 평가

> 한계: 원 논문의 손실·분산은 full-vector 시뮬레이션에 기반합니다. 공개되지 않은 FEM 데이터와 피팅 계수가 없으므로, 이 노트북은 논문에 명시된 1310/1550 nm 기준값으로 보정한 재현 가능한 해석 모델입니다.

## 논문 곡선과 정밀도 비교

그래프의 **파란색 별표 8개**는 첨부 논문 PDF의 주황색 HCF 벡터 경로를 축 보정 후 추출한 검증점입니다. 이 값들은 물리 모델의 입력이나 보정에 사용되지 않으며, 계산 완료 후 RMSE, MAPE 및 지점별 상대오차를 구할 때만 사용됩니다.


In [ ]:
%pip -q install numpy scipy matplotlib


## 실행 및 수정 방법

1. 아래 설치 셀을 실행합니다.
2. 그다음 모델 셀을 실행하면 `fig2_hcf_colab.png`가 저장되고 그래프가 표시됩니다.
3. 조건을 바꾸려면 모델 셀의 `HCFLink` 데이터클래스 값을 수정합니다. 예: `n_channels`, `span_length_km`, `transceiver_snr_db`, `nonlinear_coefficient_per_w_km`.

기본 계산 범위는 논문 Fig. 2와 동일하게 $-40$~$50$ dBm/channel입니다.

실행 결과에는 8개 검증점의 논문값·계산값·상대오차 표도 함께 출력됩니다.


In [ ]:
"""Physics-based reconstruction of the HCF curve in Fig. 2.

Target paper
------------
R. Sohanpal et al., "On the Optimum Energy-per-bit Launch Power in
Coherent Hollow-core Fibre Transmission Systems" (2026).

This script does not digitize Fig. 2 and does not contain a hand-entered
throughput curve.  For every launch power it computes, channel by channel,

    SNR_i = P_i / (P_ASE,i + eta_i P_i^3 + P_IMI,i + P_TRN,i)
    T_i   = 2 R log2(1 + SNR_i)

and sums T_i over the 29 C-band channels.  The wavelength-dependent HCF
attenuation and dispersion are supplied by the analytical DNANF model used
for the companion Fig. 1 notebook.

The exact Fig. 1 data in the paper came from a full-vector simulation and
unpublished fitting details.  The profile here is therefore a reproducible,
physics-calibrated approximation rather than the authors' original FEM data.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares


C0 = 299_792_458.0                 # vacuum speed of light [m/s]
H_PLANCK = 6.626_070_15e-34        # Planck constant [J s]
U01 = 2.4048255577                  # first zero of J0
DB_PER_NEPER = 10.0 / np.log(10.0)


# ---------------------------------------------------------------------------
# 1) Wavelength-dependent first-window DNANF attenuation and dispersion
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class DNANFGeometry:
    """Nominal first-window DNANF geometry from the companion Fig. 1 model."""

    core_radius_um: float = 14.75
    membrane_thickness_um: float = 0.50
    outer_tube_diameter_um: float = 31.05
    tube_count: int = 5
    nesting_order: int = 2

    @property
    def core_radius_m(self) -> float:
        return self.core_radius_um * 1e-6

    @property
    def membrane_thickness_m(self) -> float:
        return self.membrane_thickness_um * 1e-6

    @property
    def perimeter_gap_m(self) -> float:
        theta = np.pi / self.tube_count
        return (
            2.0 * self.core_radius_m * np.sin(theta)
            - self.outer_tube_diameter_um * 1e-6 * (1.0 - np.sin(theta))
        )


def silica_index_sellmeier(wavelength_m: np.ndarray) -> np.ndarray:
    """Fused-silica refractive index from the three-term Sellmeier law."""
    wavelength_um = np.asarray(wavelength_m, dtype=float) * 1e6
    wavelength_um_sq = wavelength_um**2
    b = np.array([0.6961663, 0.4079426, 0.8974794])
    c_um = np.array([0.0684043, 0.1162414, 9.896161])
    n_sq = np.ones_like(wavelength_um_sq)
    for bi, ci in zip(b, c_um):
        n_sq += bi * wavelength_um_sq / (wavelength_um_sq - ci**2)
    return np.sqrt(n_sq)


def hasan_radius_coefficients(geometry: DNANFGeometry) -> tuple[float, float]:
    """Geometry-driven effective-radius coefficients (Hasan et al.)."""
    radius_to_gap = geometry.core_radius_m / geometry.perimeter_gap_m
    n = geometry.tube_count
    nesting = geometry.nesting_order
    a0, a1 = 0.097041, 1.095
    b0, b1, b2, b3 = 0.76246, 0.007584, 0.002, 0.012
    f1 = a1 * np.exp(a0 / radius_to_gap)
    f2 = (
        b1 * n * np.exp(b0 / radius_to_gap)
        - b2 * n
        + b3
        + 0.0045 * np.exp(-4.1589 / (nesting * radius_to_gap))
    )
    return float(f1), float(f2)


def effective_index(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
    f1: float,
    f2: float,
) -> np.ndarray:
    """Fundamental-mode effective index from a modified capillary model."""
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    radius = geometry.core_radius_m
    wall = geometry.membrane_thickness_m
    effective_radius = f1 * radius * (1.0 - f2 * wavelength_m**2 / (radius * wall))
    return 1.0 - 0.125 * (U01 * wavelength_m / (np.pi * effective_radius)) ** 2


def chromatic_dispersion_ps_nm_km(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
    f1: float,
    f2: float,
    derivative_step_nm: float = 0.20,
) -> np.ndarray:
    """Calculate D = -(lambda/c) d2(n_eff)/d(lambda)2 with a 5-point stencil."""
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    step = derivative_step_nm * 1e-9

    def n_eff(offset: float) -> np.ndarray:
        return effective_index(wavelength_m + offset, geometry, f1, f2)

    second_derivative = (
        -n_eff(2.0 * step)
        + 16.0 * n_eff(step)
        - 30.0 * n_eff(0.0)
        + 16.0 * n_eff(-step)
        - n_eff(-2.0 * step)
    ) / (12.0 * step**2)
    dispersion_si = -(wavelength_m / C0) * second_derivative  # [s/m^2]
    return dispersion_si / 1e-6  # [ps/(nm km)]


def calibrate_dispersion(geometry: DNANFGeometry) -> tuple[float, float]:
    """Calibrate the continuous dispersion model to the two stated paper values."""
    reference_nm = np.array([1310.0, 1550.0])
    reference_d = np.array([2.17, 3.20])
    initial = np.array(hasan_radius_coefficients(geometry))

    def residual(parameters: np.ndarray) -> np.ndarray:
        calculated = chromatic_dispersion_ps_nm_km(
            reference_nm * 1e-9, geometry, parameters[0], parameters[1]
        )
        return calculated - reference_d

    result = least_squares(
        residual,
        x0=initial,
        bounds=(np.array([0.5, -0.5]), np.array([2.5, 1.0])),
        x_scale="jac",
        diff_step=1e-3,
        xtol=1e-13,
        ftol=1e-13,
        gtol=1e-13,
    )
    if not result.success or np.max(np.abs(result.fun)) > 1e-3:
        raise RuntimeError(f"Dispersion calibration failed: {result.fun}")
    return float(result.x[0]), float(result.x[1])


def capillary_bouncing_ray_loss_db_km(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
) -> np.ndarray:
    """Hybrid HE11 thin-wall bouncing-ray leakage (Bache et al.)."""
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    k0 = 2.0 * np.pi / wavelength_m
    radius = geometry.core_radius_m
    wall = geometry.membrane_thickness_m
    kappa = U01 / radius
    silica_n = silica_index_sellmeier(wavelength_m)
    sigma = k0 * np.sqrt(silica_n**2 - 1.0)
    phase = sigma * wall
    te_denominator = 4.0 * np.cos(phase) ** 2 + (
        kappa / sigma + sigma / kappa
    ) ** 2 * np.sin(phase) ** 2
    tm_denominator = 4.0 * np.cos(phase) ** 2 + (
        silica_n**2 * kappa / sigma + sigma / (silica_n**2 * kappa)
    ) ** 2 * np.sin(phase) ** 2
    alpha_te_per_m = 2.0 * U01 / (radius**2 * k0 * te_denominator)
    alpha_tm_per_m = 2.0 * U01 / (radius**2 * k0 * tm_denominator)
    return 0.5 * (alpha_te_per_m + alpha_tm_per_m) * DB_PER_NEPER * 1000.0


def attenuation_db_km(
    wavelength_m: np.ndarray,
    geometry: DNANFGeometry,
) -> np.ndarray:
    """Surface-scattering plus two-stage anti-resonant leakage model."""
    wavelength_m = np.asarray(wavelength_m, dtype=float)
    reference_nm = np.array([1550.0, 1310.0])
    reference_alpha = np.array([0.075, 0.120])
    lambda_ref_m = reference_nm[0] * 1e-9
    br_ref = capillary_bouncing_ray_loss_db_km(reference_nm * 1e-9, geometry)
    basis_reference = np.column_stack(
        [
            (lambda_ref_m / (reference_nm * 1e-9)) ** 3,
            (br_ref / br_ref[0]) ** 2,
        ]
    )
    a_surface, a_nested = np.linalg.solve(basis_reference, reference_alpha)
    if a_surface < 0.0 or a_nested < 0.0:
        raise RuntimeError("Calibrated attenuation contributions must be non-negative.")
    br = capillary_bouncing_ray_loss_db_km(wavelength_m, geometry)
    return (
        a_surface * (lambda_ref_m / wavelength_m) ** 3
        + a_nested * (br / br_ref[0]) ** 2
    )


# ---------------------------------------------------------------------------
# 2) C-band HCF transmission model used to calculate the Fig. 2 curve
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class HCFLink:
    """Parameters stated in the target paper for the Fig. 2 HCF case."""

    n_channels: int = 29
    symbol_rate_gbd: float = 140.0
    channel_spacing_ghz: float = 150.0
    c_band_min_nm: float = 1530.0
    c_band_max_nm: float = 1565.0
    span_length_km: float = 200.0
    n_spans: int = 1
    amplifier_noise_figure_db: float = 5.0
    transceiver_snr_db: float = 20.0
    nonlinear_coefficient_per_w_km: float = 5e-4
    imi_coefficient_db_per_km: float = -52.0

    @property
    def symbol_rate_hz(self) -> float:
        return self.symbol_rate_gbd * 1e9

    @property
    def channel_spacing_hz(self) -> float:
        return self.channel_spacing_ghz * 1e9

    @property
    def total_length_km(self) -> float:
        return self.n_spans * self.span_length_km


# Eight validation points extracted from the orange HCF vector path embedded
# in the paper PDF.  The PDF axes were calibrated from their vector gridlines:
# x is linear in dBm and y is logarithmic in Tb/s.  These values are displayed
# only for comparison and are never used by the physical transmission model.
PAPER_REFERENCE_LAUNCH_DBM = np.array(
    [-35.0, -20.0, -10.0, 0.0, 10.0, 22.0, 40.0, 48.0]
)
PAPER_REFERENCE_THROUGHPUT_TBPS = np.array(
    [
        1.87292462,
        21.32203450,
        41.62572367,
        50.95120740,
        52.49881133,
        52.67046429,
        42.03103403,
        10.44268748,
    ]
)


def c_band_channel_grid(link: HCFLink) -> tuple[np.ndarray, np.ndarray]:
    """Return 29 uniformly spaced C-band channel frequencies and wavelengths."""
    lower_frequency_hz = C0 / (link.c_band_max_nm * 1e-9)
    upper_frequency_hz = C0 / (link.c_band_min_nm * 1e-9)
    center_frequency_hz = 0.5 * (lower_frequency_hz + upper_frequency_hz)
    offsets = (np.arange(link.n_channels) - 0.5 * (link.n_channels - 1))
    frequency_hz = center_frequency_hz + offsets * link.channel_spacing_hz
    wavelength_m = C0 / frequency_hz
    if frequency_hz[0] < lower_frequency_hz or frequency_hz[-1] > upper_frequency_hz:
        raise ValueError("The selected channel grid does not fit inside the C-band.")
    return frequency_hz, wavelength_m


def dispersion_to_beta2_s2_per_km(
    dispersion_ps_nm_km: np.ndarray,
    wavelength_m: np.ndarray,
) -> np.ndarray:
    """Convert D [ps/(nm km)] to beta2 [s^2/km]."""
    dispersion_si = np.asarray(dispersion_ps_nm_km) * 1e-6  # [s/m^2]
    beta2_s2_per_m = -(np.asarray(wavelength_m) ** 2 / (2.0 * np.pi * C0)) * dispersion_si
    return beta2_s2_per_m * 1000.0


def closed_form_gn_eta(
    beta2_s2_per_km: np.ndarray,
    attenuation_db_per_km: np.ndarray,
    link: HCFLink,
) -> np.ndarray:
    """Closed-form GN coefficient eta [1/W^2] from Poggiolini & Poletti.

    The paper defines 2*alpha as the power-loss coefficient.  Therefore the
    field-loss coefficient used in this equation is alpha_dB / 8.685889638.
    All length quantities are consistently expressed per kilometre.
    """
    beta2 = np.abs(np.asarray(beta2_s2_per_km, dtype=float))
    alpha_field_per_km = np.asarray(attenuation_db_per_km) / 8.685889638
    gamma = link.nonlinear_coefficient_per_w_km
    rate = link.symbol_rate_hz
    spacing = link.channel_spacing_hz
    asinh_argument = (
        np.pi**2
        * beta2
        * rate**2
        / (4.0 * alpha_field_per_km)
        * link.n_channels ** (2.0 * rate / spacing)
    )
    return (
        link.n_spans
        * 4.0
        * gamma**2
        / (27.0 * np.pi * beta2 * alpha_field_per_km * rate**2)
        * np.arcsinh(asinh_argument)
    )


def per_channel_ase_w(
    frequency_hz: np.ndarray,
    attenuation_db_per_km: np.ndarray,
    link: HCFLink,
) -> np.ndarray:
    """Accumulated dual-polarisation ASE in the matched bandwidth R."""
    span_gain = 10.0 ** (
        np.asarray(attenuation_db_per_km) * link.span_length_km / 10.0
    )
    noise_factor = 10.0 ** (link.amplifier_noise_figure_db / 10.0)
    return (
        link.n_spans
        * H_PLANCK
        * np.asarray(frequency_hz)
        * noise_factor
        * link.symbol_rate_hz
        * (span_gain - 1.0)
    )


def simulate_hcf_throughput(
    link: HCFLink,
    launch_dbm: np.ndarray,
) -> dict[str, np.ndarray | float]:
    """Calculate SNR and total throughput for every requested launch power."""
    launch_dbm = np.asarray(launch_dbm, dtype=float)
    geometry = DNANFGeometry()
    f1, f2 = calibrate_dispersion(geometry)
    frequency_hz, wavelength_m = c_band_channel_grid(link)
    dispersion = chromatic_dispersion_ps_nm_km(wavelength_m, geometry, f1, f2)
    attenuation = attenuation_db_km(wavelength_m, geometry)
    beta2 = dispersion_to_beta2_s2_per_km(dispersion, wavelength_m)
    eta = closed_form_gn_eta(beta2, attenuation, link)
    ase_w = per_channel_ase_w(frequency_hz, attenuation, link)

    launch_w = 10.0 ** ((launch_dbm - 30.0) / 10.0)
    signal = launch_w[:, None]
    ase = ase_w[None, :]
    nonlinear = eta[None, :] * signal**3
    imi_ratio_per_km = 10.0 ** (link.imi_coefficient_db_per_km / 10.0)
    imi = signal * imi_ratio_per_km * link.total_length_km
    transceiver = signal / (10.0 ** (link.transceiver_snr_db / 10.0))
    snr = signal / (ase + nonlinear + imi + transceiver)
    throughput_per_channel_bps = 2.0 * link.symbol_rate_hz * np.log2(1.0 + snr)
    total_throughput_tbps = throughput_per_channel_bps.sum(axis=1) / 1e12
    optimum_index = int(np.argmax(total_throughput_tbps))

    if not np.all(np.isfinite(total_throughput_tbps)) or np.any(total_throughput_tbps <= 0.0):
        raise RuntimeError("The calculated throughput contains invalid values.")

    return {
        "launch_dbm": launch_dbm,
        "launch_w": launch_w,
        "frequency_hz": frequency_hz,
        "wavelength_m": wavelength_m,
        "attenuation_db_km": attenuation,
        "dispersion_ps_nm_km": dispersion,
        "beta2_s2_km": beta2,
        "eta_per_w2": eta,
        "ase_w": ase_w,
        "snr_linear": snr,
        "throughput_per_channel_bps": throughput_per_channel_bps,
        "total_throughput_tbps": total_throughput_tbps,
        "optimum_index": optimum_index,
        "optimum_launch_dbm": float(launch_dbm[optimum_index]),
        "maximum_throughput_tbps": float(total_throughput_tbps[optimum_index]),
    }


def compare_with_paper(
    result: dict[str, np.ndarray | float],
) -> dict[str, np.ndarray | float]:
    """Compare the calculation with eight vector-extracted Fig. 2 points."""
    model_at_reference = np.interp(
        PAPER_REFERENCE_LAUNCH_DBM,
        np.asarray(result["launch_dbm"]),
        np.asarray(result["total_throughput_tbps"]),
    )
    error_tbps = model_at_reference - PAPER_REFERENCE_THROUGHPUT_TBPS
    relative_error_percent = 100.0 * error_tbps / PAPER_REFERENCE_THROUGHPUT_TBPS
    return {
        "launch_dbm": PAPER_REFERENCE_LAUNCH_DBM.copy(),
        "paper_tbps": PAPER_REFERENCE_THROUGHPUT_TBPS.copy(),
        "model_tbps": model_at_reference,
        "error_tbps": error_tbps,
        "relative_error_percent": relative_error_percent,
        "rmse_tbps": float(np.sqrt(np.mean(error_tbps**2))),
        "mape_percent": float(np.mean(np.abs(relative_error_percent))),
        "maximum_absolute_relative_error_percent": float(
            np.max(np.abs(relative_error_percent))
        ),
    }


def plot_hcf_curve(
    result: dict[str, np.ndarray | float],
    output_path: Path | str = Path("fig2_hcf_colab.png"),
) -> plt.Figure:
    """Plot only the orange HCF curve using the axes of the paper's Fig. 2."""
    launch_dbm = np.asarray(result["launch_dbm"])
    throughput = np.asarray(result["total_throughput_tbps"])
    optimum_index = int(result["optimum_index"])
    comparison = compare_with_paper(result)

    fig, ax = plt.subplots(figsize=(8.2, 5.5), constrained_layout=True)
    ax.plot(
        launch_dbm,
        throughput,
        color="#e66100",
        lw=2.6,
        label="Physics model (this code)",
    )
    ax.plot(
        launch_dbm[optimum_index],
        throughput[optimum_index],
        marker="s",
        ms=7,
        color="#e66100",
        markeredgecolor="white",
        markeredgewidth=0.8,
        zorder=5,
    )
    ax.scatter(
        comparison["launch_dbm"],
        comparison["paper_tbps"],
        marker="*",
        s=115,
        color="#1666b1",
        edgecolors="white",
        linewidths=0.7,
        label="Paper Fig. 2 (8 vector-extracted points)",
        zorder=6,
    )
    ax.annotate(
        rf"$T_{{\rm tot}}^{{\max}}$ = {throughput[optimum_index]:.2f} Tb/s"
        "\n"
        rf"$P_{{\rm ch}}$ = {launch_dbm[optimum_index]:.1f} dBm",
        xy=(launch_dbm[optimum_index], throughput[optimum_index]),
        xytext=(-105, -46),
        textcoords="offset points",
        fontsize=10.5,
        arrowprops={"arrowstyle": "->", "color": "#555555", "lw": 1.0},
    )
    ax.set_yscale("log")
    ax.set_xlim(-40.0, 50.0)
    ax.set_ylim(0.8, 70.0)
    ax.set_xticks([-40, -20, 0, 20, 40])
    ax.set_yticks([1, 10, 60], labels=["1", "10", "60"])
    ax.set_xlabel("Launch power per channel (dBm)", fontsize=12)
    ax.set_ylabel("C-band throughput (Tb/s)", fontsize=12)
    ax.set_title("Fig. 2 HCF reconstruction — physics-based calculation", fontsize=13)
    ax.grid(True, which="major", color="#8a8a8a", alpha=0.40, linewidth=0.8)
    ax.grid(True, which="minor", axis="y", color="#b5b5b5", alpha=0.16, linewidth=0.5)
    ax.text(
        0.025,
        0.965,
        f"RMSE = {comparison['rmse_tbps']:.3f} Tb/s\n"
        f"MAPE = {comparison['mape_percent']:.2f}%",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9.5,
        bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "#888888", "alpha": 0.92},
    )
    ax.legend(loc="lower right", frameon=True, framealpha=0.95, fontsize=9.5)
    output_path = Path(output_path)
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    return fig


def print_summary(result: dict[str, np.ndarray | float], link: HCFLink) -> None:
    """Print the main model outputs and basic reproduction checks."""
    wavelength_nm = np.asarray(result["wavelength_m"]) * 1e9
    attenuation = np.asarray(result["attenuation_db_km"])
    dispersion = np.asarray(result["dispersion_ps_nm_km"])
    eta = np.asarray(result["eta_per_w2"])
    ase_w = np.asarray(result["ase_w"])
    optimum_launch_dbm = float(result["optimum_launch_dbm"])
    maximum_throughput_tbps = float(result["maximum_throughput_tbps"])

    print("=== Fig. 2 HCF calculation ===")
    print(f"Channels / baud rate / spacing : {link.n_channels} / {link.symbol_rate_gbd:.0f} GBd / {link.channel_spacing_ghz:.0f} GHz")
    print(f"C-band channel wavelengths     : {wavelength_nm.min():.2f}–{wavelength_nm.max():.2f} nm")
    print(f"HCF attenuation across grid    : {attenuation.min():.4f}–{attenuation.max():.4f} dB/km")
    print(f"HCF dispersion across grid     : {dispersion.min():.4f}–{dispersion.max():.4f} ps/(nm·km)")
    print(f"GN eta across grid             : {eta.min():.3e}–{eta.max():.3e} 1/W²")
    print(f"ASE per channel                : {10.0 * np.log10(ase_w.mean()) + 30.0:.2f} dBm (mean)")
    print(f"Maximum throughput             : {maximum_throughput_tbps:.3f} Tb/s")
    print(f"Launch power at maximum        : {optimum_launch_dbm:.2f} dBm/channel")

    comparison = compare_with_paper(result)
    print("\n=== Comparison with paper's vector curve ===")
    print(" Pch (dBm) | Paper (Tb/s) | Model (Tb/s) | Error (%)")
    print("------------+--------------+--------------+----------")
    for launch, paper, model, error_percent in zip(
        comparison["launch_dbm"],
        comparison["paper_tbps"],
        comparison["model_tbps"],
        comparison["relative_error_percent"],
    ):
        print(f" {launch:10.1f} | {paper:12.4f} | {model:12.4f} | {error_percent:+8.3f}")
    print(f"RMSE                            : {comparison['rmse_tbps']:.4f} Tb/s")
    print(f"Mean absolute percentage error  : {comparison['mape_percent']:.3f}%")
    print(
        "Maximum absolute relative error: "
        f"{comparison['maximum_absolute_relative_error_percent']:.3f}%"
    )

    # Expected ranges are broad physical/reproduction checks, not curve inputs.
    assert 50.0 < maximum_throughput_tbps < 55.0
    assert 15.0 < optimum_launch_dbm < 30.0
    print("Validation: PASS (finite curve and expected Fig. 2 HCF range)")


def main() -> tuple[dict[str, np.ndarray | float], plt.Figure]:
    link = HCFLink()
    launch_dbm = np.linspace(-40.0, 50.0, 1201)
    result = simulate_hcf_throughput(link, launch_dbm)
    figure = plot_hcf_curve(result, Path("fig2_hcf_colab.png"))
    print_summary(result, link)
    plt.show()
    return result, figure


if __name__ == "__main__":
    result, figure = main()


## 모델 참고문헌

1. R. Sohanpal *et al.*, “On the Optimum Energy-per-bit Launch Power in Coherent Hollow-core Fibre Transmission Systems,” 2026.
2. P. Poggiolini and F. Poletti, “Opportunities and Challenges for Long-Distance Transmission in Hollow-Core Fibres,” *J. Lightwave Technol.*, 40(6), 1605–1616 (2022), DOI: 10.1109/JLT.2021.3140114.
3. M. I. Hasan *et al.*, “Analytical model of the effective modal refractive index of the LP01 mode of non-ideal hollow-core anti-resonant fibers,” arXiv:1708.06879.


## 독립 검증에 사용하는 규칙

기존 Fig. 2의 `calibrate_dispersion()`과 `attenuation_db_km()`는 Sohanpal Fig. 2 재현을 위한
보정 모델이다. Petrovich 독립검증에서는 이를 호출하지 않고 `hasan_radius_coefficients`,
`chromatic_dispersion_ps_nm_km`, `capillary_bouncing_ray_loss_db_km`의 Raw 결과만 사용한다.

손실 비교는 geometry-only leakage proxy와 측정된 총손실을 비교하는 진단이다. Petrovich의
총손실을 재현하려면 FEM leakage, surface scattering, microbend, gas absorption을 합산해야 한다.


In [ ]:
from scipy.special import erfc, erfcinv
from scipy.optimize import minimize_scalar
from dataclasses import dataclass
import json, platform, pathlib, pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj): print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) else obj)

VALIDATION_RESULTS = pathlib.Path('validation_0905_results')
VALIDATION_RESULTS.mkdir(exist_ok=True)
VALIDATION_TABLES = {}
VALIDATION_FIGURES = []

def save_table_0905(name, frame):
    VALIDATION_TABLES[name] = frame.copy()
    frame.to_csv(VALIDATION_RESULTS/(name+'.csv'), index=False)
    display(frame)

def save_fig_0905(name, fig):
    fig.savefig(VALIDATION_RESULTS/(name+'.png'), dpi=180, bbox_inches='tight')
    fig.savefig(VALIDATION_RESULTS/(name+'.svg'), bbox_inches='tight')
    VALIDATION_FIGURES.append(name)
    plt.show()

def error_metrics_0905(model, reference):
    model=np.asarray(model,dtype=float); reference=np.asarray(reference,dtype=float)
    delta=model-reference
    ape=np.abs(100*delta/reference)
    return {'MAPE_percent':float(np.mean(ape)),
            'RMSE':float(np.sqrt(np.mean(delta**2))),
            'MAE':float(np.mean(np.abs(delta))),
            'MAX_APE_percent':float(np.max(ape))}
print('0905 validation initialized; no paper-output fitting is enabled.')


## 1. Poggiolini GN model

동일한 fiber 입력으로 기존 closed-form eta와 독립적으로 작성한 유한 span Eq. 13을 비교한다.
이는 수식 구현 검산이며 SSFM 실험 검증과 구분한다.


In [ ]:
@dataclass(frozen=True)
class GNFiber0905:
    name: str
    alpha_db_km: float
    D_ps_nm_km: float
    gamma_W_inv_km: float

def beta2_0905(D, wavelength_nm=1550.0):
    return abs(-(wavelength_nm*1e-9)**2/(2*np.pi*C0)*D*1e-6*1000)

def dbm2w_0905(x):
    return 1e-3*10**(np.asarray(x,dtype=float)/10)

def eta_legacy_0905(f, Rs, spacing, Nch, Nsp=1):
    b2=beta2_0905(f.D_ps_nm_km)
    a=f.alpha_db_km/8.685889638
    arg=np.pi**2*b2*Rs**2/(4*a)*Nch**(2*Rs/spacing)
    return Nsp*4*f.gamma_W_inv_km**2/(27*np.pi*b2*a*Rs**2)*np.arcsinh(arg)

def eta_eq13_0905(f, Rs, spacing, Nch, Ls):
    b2=beta2_0905(f.D_ps_nm_km)
    a=f.alpha_db_km/8.685889638
    Leff=(1-np.exp(-2*a*Ls))/(2*a)
    Linf=1/(2*a)
    arg=.5*np.pi**2*b2*Linf*(Nch*Rs)**2
    return 8/27*f.gamma_W_inv_km**2*Leff**2/(np.pi*b2*Linf*Rs**2)*np.arcsinh(arg)

GN0905=[GNFiber0905('LPSCF',.165,20.4,.8),GNFiber0905('SMF',.2,16.5,1.3),GNFiber0905('NZDSF',.2,3.9,1.6)]
gn_rows=[]
for f in GN0905:
    legacy=eta_legacy_0905(f,32e9,32e9,157)
    eq13=eta_eq13_0905(f,32e9,32e9,157,100)
    gn_rows.append({'fiber':f.name,'eq13_eta_W_inv2':eq13,'legacy_eta_W_inv2':legacy,
                    'absolute_error_percent':abs(100*(legacy-eq13)/eq13)})
gn0905_df=pd.DataFrame(gn_rows)
save_table_0905('01_poggiolini_eta_comparison',gn0905_df)
gn0905_metrics=error_metrics_0905(gn0905_df.legacy_eta_W_inv2,gn0905_df.eq13_eta_W_inv2)
fig,ax=plt.subplots(figsize=(8,4.5),layout='constrained')
x=np.arange(len(gn0905_df)); ax.bar(x-.2,gn0905_df.eq13_eta_W_inv2,.4,label='Eq.13',color='#167b83'); ax.bar(x+.2,gn0905_df.legacy_eta_W_inv2,.4,label='Existing closed form',color='#d85b43')
ax.set_xticks(x,gn0905_df.fiber); ax.set_ylabel('eta (1/W²)'); ax.set_title(f'Poggiolini GN eta — MAPE {gn0905_metrics["MAPE_percent"]:.2f}%'); ax.legend(); save_fig_0905('01_poggiolini_eta',fig)


## 2. Nespola PM-16QAM 7종 fiber 검증

Table I의 fiber parameter와 시스템 조건을 그대로 입력한다. 논문은 실제 back-to-back BER–OSNR
곡선을 사용하므로, 아래 결과는 그 곡선이 공개되지 않은 상태에서의 독립 재계산이다.


In [ ]:
NESPOLA0905=pd.DataFrame([
 ['SSMF',.190,75.,1.26,16.84,51.06,1940.,0.0],['NZDSF',.200,43.,2.00,2.58,50.18,602.,2.0],
 ['PSCF80',.164,86.,1.04,16.36,54.44,2395.,.3],['PSCF110',.161,111.,.81,20.50,53.18,3084.,.4],
 ['PSCF130',.162,131.,.68,20.92,54.42,3374.,.6],['PSCF150',.161,150.,.59,20.69,54.44,3810.,.6],
 ['DCF',.457,16.8,6.03,-166.47,20.11,502.,2.5]],
 columns=['fiber','alpha_db_km','Aeff_um2','gamma_W_inv_km','D_ps_nm_km','span_km','paper_Lmax_km','splice_out_db'])
NESPOLA_SYS0905={'Rs_hz':15.625e9,'spacing_hz':16e9,'Nch':22,'NF_dB':5.5,'BER_target':1.5e-2,'B2B_penalty_dB':2.5}
save_table_0905('02_nespola_inputs',NESPOLA0905)


In [ ]:
def eta_nespola_0905(row, pm_power='total'):
    Rs=NESPOLA_SYS0905['Rs_hz']; df=NESPOLA_SYS0905['spacing_hz']; Nch=NESPOLA_SYS0905['Nch']
    b2=beta2_0905(row.D_ps_nm_km); a=row.alpha_db_km/8.685889638
    Leff=(1-np.exp(-2*a*row.span_km))/(2*a); Linf=1/(2*a)
    # PM total channel convention: expose the factor instead of hiding it.
    coeff=16/27 if pm_power=='total' else 8/27
    arg=np.pi**2*b2*Linf*Rs**2/(4*a)*Nch**(2*Rs/df)
    return coeff*row.gamma_W_inv_km**2*Leff**2/(np.pi*b2*Linf*Rs**2)*np.arcsinh(arg)

def nespola_reach_0905(row, pm_power='total', b2b_curve=None):
    Rs=NESPOLA_SYS0905['Rs_hz']; lam=1550e-9
    Bn=C0/lam**2*0.1e-9
    NF=10**(NESPOLA_SYS0905['NF_dB']/10)
    gain_db=row.alpha_db_km*row.span_km+row.splice_out_db
    pase=H_PLANCK*C0/lam*NF*Bn*(10**(gain_db/10)-1)
    eta=eta_nespola_0905(row,pm_power)
    if b2b_curve is None:
        threshold=10*erfcinv(8*NESPOLA_SYS0905['BER_target']/3)**2
        threshold*=10**(NESPOLA_SYS0905['B2B_penalty_dB']/10)
    else:
        threshold=10**(np.interp(np.log10(NESPOLA_SYS0905['BER_target']),np.log10(b2b_curve.ber[::-1]),b2b_curve.osnr_db[::-1])/10)
    grid=np.linspace(-10,8,721); best=None
    for pdbm in grid:
        p=dbm2w_0905(pdbm); inv=pase/p+eta*p*p
        ns=max(0,int(np.floor(1/(threshold*inv))))
        reach=ns*row.span_km
        if best is None or reach>best['reach_km']:
            best={'fiber':row.fiber,'launch_dBm':pdbm,'eta_W_inv2':eta,'PASE_W':pase,'threshold_dB':10*np.log10(threshold),'Nspans':ns,'reach_km':reach,'paper_Lmax_km':row.paper_Lmax_km,'error_percent':abs(100*(reach-row.paper_Lmax_km)/row.paper_Lmax_km)}
    return best

nespola_results=pd.DataFrame([nespola_reach_0905(r) for _,r in NESPOLA0905.iterrows()])
save_table_0905('03_nespola_lmax_comparison',nespola_results)
nespola_metrics=error_metrics_0905(nespola_results.reach_km,nespola_results.paper_Lmax_km)
print('Nespola MAPE:',nespola_metrics)
fig,ax=plt.subplots(1,2,figsize=(12,4.5),layout='constrained'); x=np.arange(len(nespola_results))
ax[0].bar(x-.2,nespola_results.paper_Lmax_km,.4,label='Paper experiment',color='#167b83'); ax[0].bar(x+.2,nespola_results.reach_km,.4,label='Code reconstruction',color='#d85b43'); ax[0].set(yscale='log',xticks=x,xticklabels=nespola_results.fiber,ylabel='Lmax (km)',title=f'Nespola — MAPE {nespola_metrics["MAPE_percent"]:.2f}%'); ax[0].legend(fontsize=8)
ax[1].bar(nespola_results.fiber,nespola_results.error_percent,color='#d85b43'); ax[1].axhline(10,color='gray',ls=':'); ax[1].set(ylabel='Absolute error (%)',title='Pointwise error'); save_fig_0905('02_nespola_lmax',fig)


## 3. Carena 2012 첨부 그림

Carena 논문은 PM-BPSK, PM-QPSK, PM-8QAM, PM-16QAM의 시뮬레이션 결과를 제시하지만,
현재 확보된 자료에는 패널별 정확한 fiber parameter, span, NF, BER/FEC, marker 정의가 모두 없다.
따라서 임의로 곡선을 만들지 않고 `INPUTS_UNVERIFIED`로 표시한다. 원 PDF와 입력표를 넣으면
동일한 `compare_curve()` 인터페이스에서 MAPE/RMSE를 계산할 수 있다.


In [ ]:
CARENA_STATUS={'paper':'Carena et al. JLT 30(10), 1524-1539 (2012)','status':'INPUTS_UNVERIFIED',
 'reason':'Exact panel inputs and simulation marker identity are not available in the current attachment.'}
(VALIDATION_RESULTS/'04_carena_status.json').write_text(json.dumps(CARENA_STATUS,ensure_ascii=False,indent=2))
print(CARENA_STATUS)


## 4. Petrovich HCF geometry 분산·손실 검증

Petrovich의 1310/1550/1700 nm 분산은 FEM 모델값이고, 1310/1550 nm 손실은 실측 총손실이다.
Raw geometry는 보정하지 않고 비교한다. 손실의 경우 raw bouncing-ray는 leakage proxy일 뿐이므로
총손실의 독립 검증으로 해석하지 않는다.


In [ ]:
geometry0905=DNANFGeometry()
wlD0905=np.array([1310.,1550.,1700.]); rawD0905=chromatic_dispersion_ps_nm_km(wlD0905*1e-9,geometry0905,*hasan_radius_coefficients(geometry0905))
refD0905=np.array([2.1,3.2,3.7])
disp0905=pd.DataFrame({'wavelength_nm':wlD0905,'paper_FEM':refD0905,'raw_geometry':rawD0905,'absolute_error_percent':abs(100*(rawD0905-refD0905)/refD0905)})
save_table_0905('05_petrovich_dispersion',disp0905)
wlL0905=np.array([1310.,1550.]); rawL0905=capillary_bouncing_ray_loss_db_km(wlL0905*1e-9,geometry0905); refL0905=np.array([.128,.091])
loss0905=pd.DataFrame({'wavelength_nm':wlL0905,'paper_measured_total_db_km':refL0905,'raw_leakage_proxy_db_km':rawL0905,'absolute_error_percent':abs(100*(rawL0905-refL0905)/refL0905)})
save_table_0905('06_petrovich_loss_layer_audit',loss0905)
print('Dispersion metrics:',error_metrics_0905(rawD0905,refD0905)); print('Loss metrics:',error_metrics_0905(rawL0905,refL0905))
fig,ax=plt.subplots(1,2,figsize=(11,4.3),layout='constrained'); ax[0].plot(wlD0905,refD0905,'o-',label='Petrovich FEM'); ax[0].plot(wlD0905,rawD0905,'s--',label='Raw Hasan/MS'); ax[0].set(xlabel='Wavelength (nm)',ylabel='D (ps/nm/km)',title='Dispersion'); ax[0].legend()
ax[1].bar(wlL0905-12,refL0905,24,label='Measured total'); ax[1].bar(wlL0905+12,rawL0905,24,label='Raw proxy'); ax[1].set_yscale('log'); ax[1].set(xlabel='Wavelength (nm)',ylabel='Loss (dB/km)',title='Loss model-layer audit'); ax[1].legend(); save_fig_0905('03_petrovich_audit',fig)


## 5. 수정 방향과 연구 사용 판정

Nespola의 오차를 줄이려면 Eq.18의 spacing 항, PM 전력 convention, 0.1-nm OSNR, 실제 B2B BER 곡선을
반영해야 한다. Petrovich 손실은 단순 bouncing-ray 식을 조정하지 않고, FEM 복소 $n_{eff}$에서 leakage를
구한 뒤 surface scattering·microbend·gas absorption을 더해야 한다. 이 네 계수는 15개 DNANF 데이터로
학습하고 별도 DNANF로 hold-out 검증해야 한다.


In [ ]:
paper_status_0905=pd.DataFrame([
 ['Poggiolini 2012','eta','Eq.13 consistency',float(gn0905_metrics['MAPE_percent']),'PASS'],
 ['Nespola 2014','PM-16QAM Lmax','GN + AWGN/B2B approximation',float(nespola_metrics['MAPE_percent']),'PASS_WITH_MODEL_ASSUMPTIONS'],
 ['Carena 2012','4-format figure','inputs unavailable',np.nan,'INPUTS_UNVERIFIED'],
 ['Petrovich 2025','dispersion','Raw geometry vs FEM',float(error_metrics_0905(rawD0905,refD0905)['MAPE_percent']),'MODEL_VS_FEM'],
 ['Petrovich 2025','loss','Raw leakage proxy vs measured total',float(error_metrics_0905(rawL0905,refL0905)['MAPE_percent']),'MODEL_LAYER_MISMATCH']],
 columns=['paper','output','comparison','MAPE_percent','status'])
save_table_0905('07_validation_status',paper_status_0905)
summary0905={'status':paper_status_0905.to_dict(orient='records'),'runtime':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__},'figures':VALIDATION_FIGURES}
(VALIDATION_RESULTS/'summary.json').write_text(json.dumps(summary0905,ensure_ascii=False,indent=2))
print(json.dumps(summary0905,ensure_ascii=False,indent=2))
